# Guardrails Kit — démonstration

Ce notebook montre le format actuel : une configuration YAML, des sorties nommées pour les transforms, puis des vérifications séparées. Il termine par un exemple de construction Python sans YAML.

Exécute les cellules dans l’ordre. Les étapes de classification et de similarité appellent le modèle défini dans `.env`.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "examples" / "models.py").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not find examples/models.py")
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from examples.models import MessageClassification
from openai import OpenAI
from dotenv import load_dotenv
from guardrails_kit import (
    CheckStep,
    CheckOperation,
    CheckResult,
    CheckStatus,
    Guardrails,
    PIIRedactor,
    TransformStep,
    TransformOperation,
)

## 1. Charger la configuration

Le fichier `.env` du projet contient `OPENAI_BASE_URL`, `OPENAI_API_KEY` et `OPENAI_MODEL`. `OPENAI_EMBEDDING_MODEL` peut aussi être défini pour le check de similarité.

In [2]:
load_dotenv(project_root / ".env", override=False)
client = OpenAI()
guardrails = Guardrails.from_yaml(
    project_root / "examples" / "guardrails.yaml",
    openai_client=client,
)
print("API base URL:", client.base_url)
print("Configuration chargée.")

API base URL: http://localhost:11434/v1/
Configuration chargée.


## 2. Transformer et vérifier une entrée

La PII est masquée avant l’extraction. Le texte nettoyé et la classification sont accessibles par leur nom dans `report.outputs`.

In [7]:
payload = {
    "customer": {
        "name": "Ada Lovelace",
        "email": "ada@example.com",
        "age": 36,
        "message": (
            "My name is Ada Lovelace. Contact me at ada@example.com. "
            "I was charged twice for my monthly subscription."
        ),
    }
}

report = guardrails.validate_input(payload)
print(
    f"passed={report.passed} | failures={report.has_failures} | errors={report.has_errors}"
)

print("Transforms")
for item in report.transforms:
    print(f"  {item.status.value:7} {item.name}: {item.message}")

print("Checks")
for item in report.checks:
    print(f"  {item.status.value:7} {item.name}: {item.message}")

safe_message = report.get_output("safe_message", str)
classification = report.get_output("classification", MessageClassification)
assert "ada@example.com" not in safe_message
print("\nTexte nettoyé :", safe_message)
print("Classification typée :", classification)

passed=False | failures=True | errors=False
Transforms
  success redact_customer_message: Transformed str into outputs.safe_message.
  success extract_message_classification: Transformed MessageClassification into outputs.classification.
Checks
  pass    customer_matches_schema: Data matches Customer.
  fail    message_is_support_related: Similarity 0.7242 is below threshold 0.7800; closest match is example 1.
  pass    extracted_summary_is_not_empty: Value satisfies condition not_equals ''.

Texte nettoyé : My name is [PERSON_NAME]. Contact me at [EMAIL]. I was charged twice for my monthly subscription.
Classification typée : category='technical' urgency='high' summary='The user is experiencing two charges for their monthly subscription, which they have not logged out or registered for beforehand.'


## 3. Vérifier une réponse

Les checks de la phase `output` valident le schéma de réponse et bloquent les clés qui correspondent au motif configuré.

In [4]:
answer = {"answer": "Your subscription has been corrected.", "confidence": 0.95}
output_report = guardrails.validate_output(answer)
for item in output_report.checks:
    print(f"{item.status.value:7} {item.name}: {item.message}")
print("Réponse acceptée :", output_report.passed)

pass    reply_matches_schema: Data matches SupportReply.
pass    reply_has_no_api_key: Field '<input>' does not contain forbidden pattern '\\bsk-[A-Za-z0-9_-]{16,}\\b' (mode=search).
Réponse acceptée : True


## 4. Créer et tester un transform personnalisé en Python

Cette étape hérite de `TransformOperation` et implémente `transform`. Le pipeline la place après le redactor PII et vérifie sa sortie.


In [12]:
class NormalizeWhitespace(TransformOperation):
    type_name = "normalize_whitespace"

    def transform(self, value: str) -> str:
        return " ".join(value.split())


class MessageIsNonEmptyCheck(CheckOperation):
    type_name = "message_is_non_empty"

    def __init__(self):
        super().__init__("message_is_non_empty")

    def check(self, text: str) -> CheckResult:
        is_non_empty = bool(text.strip())
        return self._result(
            CheckStatus.PASS if is_non_empty else CheckStatus.FAIL,
            "Message is non-empty." if is_non_empty else "Message is empty.",
        )


python_guardrails = Guardrails(
    [
        TransformStep(
            name="redact_message",
            source="input.message",
            target="safe_message",
            using=PIIRedactor(),
        ),
        TransformStep(
            name="normalize_whitespace",
            source="outputs.safe_message",
            target="normalized_message",
            using=NormalizeWhitespace(),
        ),
        CheckStep(
            name="message_is_non_empty",
            source="outputs.normalized_message",
            using=MessageIsNonEmptyCheck(),
        ),
    ]
)

python_report = python_guardrails.validate_input(
    {"message": "Je m'appelle Ada Lovelace,   écrivez à ada@example.com."}
)
safe_message = python_report.get_output("safe_message", str)
normalized_message = python_report.get_output("normalized_message", str)
assert "ada@example.com" not in safe_message
assert "  " not in normalized_message
print("Texte masqué :", safe_message)
print("Texte normalisé :", normalized_message)
print("Résultat transform :", python_report.transforms[1].status.value)
print("Résultat check :", python_report.checks[0].status.value)

Texte masqué : Je m'appelle [PERSON_NAME],   écrivez à [EMAIL].
Texte normalisé : Je m'appelle [PERSON_NAME], écrivez à [EMAIL].
Résultat transform : success
Résultat check : pass


## Résumé sérialisable

En Python, les modèles extraits restent des objets typés. Pour journaliser ou transmettre le rapport, utilisez `to_dict()`.

In [6]:
summary = report.to_dict()
print(
    {
        "phase": summary["phase"],
        "passed": summary["passed"],
        "transform_count": len(summary["transforms"]),
        "check_count": len(summary["checks"]),
        "output_names": list(summary["outputs"]),
    }
)

{'phase': 'input', 'passed': False, 'transform_count': 2, 'check_count': 3, 'output_names': ['safe_message', 'classification']}
